# Aula 3 — Regressão Linear com um Regressor

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

O instrumento básico do curso: o modelo

$$Y_i = \beta_0 + \beta_1 X_i + u_i$$

e o estimador de **mínimos quadrados ordinários**. Quatro resultados:

1. MQO **na mão** contra `lm()` — a fórmula $\hat\beta_1 = s_{XY}/s_X^2$;
2. as **condições de ortogonalidade** que definem o estimador;
3. a decomposição $TSS = ESS + SSR$ e as três formas do $R^2$;
4. a **distribuição amostral** de $\hat\beta_1$, por simulação.

> ⚠️ Os dados do exemplo *TestScore* × *STR* são **simulados e calibrados** para se
> parecerem com os de Stock & Watson (2020) — não são os dados originais. A vantagem
> é que aqui **conhecemos** os parâmetros verdadeiros.

In [ ]:
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))
theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)

## 1. Os dados e a regressão

In [ ]:
set.seed(5490)
n_d <- 420
beta0_v <- 698.9; beta1_v <- -2.28   # parâmetros VERDADEIROS

STR       <- rnorm(n_d, mean = 19.64, sd = 1.89)
TestScore <- beta0_v + beta1_v * STR + rnorm(n_d, 0, 18.6)
ca <- data.frame(TestScore, STR)

m1 <- lm(TestScore ~ STR, data = ca)
summary(m1)$coefficients

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 4.2)
ggplot(ca, aes(STR, TestScore)) +
  geom_point(alpha = 0.45, colour = cz, size = 1.4) +
  geom_smooth(method = "lm", se = FALSE, colour = vm, linewidth = 1.1) +
  labs(x = "razão aluno-professor (STR)", y = "nota média do distrito",
       subtitle = sprintf("reta de MQO: intercepto %.1f, inclinação %.3f",
                          coef(m1)[1], coef(m1)[2]))

## 2. MQO na mão

O estimador é a **covariância sobre a variância**:

$$\hat\beta_1 = \frac{s_{XY}}{s_X^2}, \qquad \hat\beta_0 = \bar{Y} - \hat\beta_1\bar{X}$$

In [ ]:
b1_mao <- cov(ca$STR, ca$TestScore) / var(ca$STR)
b0_mao <- mean(ca$TestScore) - b1_mao * mean(ca$STR)

rbind(na_mao = c(b0 = b0_mao, b1 = b1_mao),
      lm     = c(coef(m1)[1], coef(m1)[2]),
      verdade = c(beta0_v, beta1_v)) |> round(6)

## 3. As condições de ortogonalidade

O MQO é definido por duas condições sobre os resíduos: eles **somam zero** e são
**ortogonais ao regressor**. Não é resultado empírico — é consequência algébrica das
condições de primeira ordem.

In [ ]:
u_hat <- residuals(m1)
c(soma_residuos        = sum(u_hat),
  soma_x_vezes_residuo = sum(ca$STR * u_hat),
  cor_x_residuo        = cor(ca$STR, u_hat)) |> round(10)

In [ ]:
# a reta passa pelo ponto das médias
c(media_X = mean(ca$STR), media_Y = mean(ca$TestScore),
  previsto_na_media = coef(m1)[1] + coef(m1)[2] * mean(ca$STR)) |> round(6)

## 4. TSS = ESS + SSR, e as três formas do $R^2$

In [ ]:
y     <- ca$TestScore
y_hat <- fitted(m1)

TSS <- sum((y - mean(y))^2)
ESS <- sum((y_hat - mean(y))^2)
SSR <- sum(u_hat^2)

c(TSS = TSS, ESS = ESS, SSR = SSR,
  ESS_mais_SSR = ESS + SSR, diferenca = TSS - (ESS + SSR)) |> round(6)

In [ ]:
# as três formas equivalentes do R²
c(um_menos_SSR_TSS = 1 - SSR / TSS,
  ESS_sobre_TSS    = ESS / TSS,
  cor_ao_quadrado  = cor(y, y_hat)^2,
  do_summary       = summary(m1)$r.squared) |> round(8)

In [ ]:
# SER: o desvio-padrão dos resíduos, corrigido por graus de liberdade
c(SER          = sqrt(SSR / (n_d - 2)),
  sigma_do_lm  = summary(m1)$sigma,
  sd_verdadeiro = 18.6) |> round(4)

O $R^2$ é baixo — o tamanho da turma explica pouco da variação das notas. **Isso não
é um problema para a estimativa causal**: ajuste e identificação são coisas diferentes,
ponto que a Aula 6 desenvolve.

## 5. A distribuição amostral de $\hat\beta_1$

O estimador é uma **variável aleatória**: muda de amostra para amostra. Vamos ver sua
distribuição repetindo o experimento 2.000 vezes.

In [ ]:
set.seed(31)
M <- 2000
betas <- replicate(M, {
  x <- rnorm(n_d, 19.64, 1.89)
  yy <- beta0_v + beta1_v * x + rnorm(n_d, 0, 18.6)
  coef(lm(yy ~ x))[2]
})

c(verdadeiro     = beta1_v,
  media_estimada = mean(betas),
  vies           = mean(betas) - beta1_v,
  desvio_padrao  = sd(betas),
  ep_teorico     = 18.6 / (sqrt(n_d) * 1.89)) |> round(4)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 3.8)
ggplot(data.frame(betas), aes(betas)) +
  geom_histogram(aes(y = after_stat(density)), bins = 45,
                 fill = az, alpha = 0.55, colour = NA) +
  stat_function(fun = dnorm, args = list(mean = mean(betas), sd = sd(betas)),
                colour = vm, linewidth = 1) +
  geom_vline(xintercept = beta1_v, colour = vd, linetype = 2, linewidth = 0.9) +
  labs(x = expression(hat(beta)[1]), y = "densidade",
       subtitle = "verde tracejado: valor verdadeiro · vermelho: Normal ajustada")

A distribuição é **centrada no valor verdadeiro** (não viesada) e visivelmente
**Normal**, como o Teorema Central do Limite prevê — e o desvio-padrão simulado bate
com a fórmula teórica.

## 6. O que uma única observação extrema pode fazer

In [ ]:
ca_out <- rbind(ca, data.frame(TestScore = 850, STR = 30))  # 1 ponto em 421
m_out  <- lm(TestScore ~ STR, data = ca_out)

rbind(sem_outlier = coef(m1), com_outlier = coef(m_out)) |> round(3)

Uma observação em 421 desloca a inclinação de forma perceptível. Daí a recomendação
de **sempre olhar o gráfico** antes de confiar num coeficiente.

## Para experimentar

1. Reduza `n_d` para 40 e refaça a simulação: o estimador continua não viesado, mas
   o desvio-padrão cresce — separe **viés** de **precisão**.
2. Aumente o ruído (`sd = 40`) e veja o $R^2$ despencar sem que $\hat\beta_1$ deixe de
   ser não viesado.
3. Mova o outlier para `STR = 19.6` (no centro dos dados): o estrago é menor. Por quê?

---

⬅️ [Aula 2](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/02-probabilidade.ipynb) · [🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb) · ➡️ [**Aula 4 — Inferência**](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/04-inferencia.ipynb)